<a href="https://colab.research.google.com/github/itsrealfarman/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/itsrealfarman/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

Paper audited: **"The State of AI-Driven SEO — FlyRank Data Report, April 2026"**.

**Working month:** `2026-03`, split at `2026-03-16` — same setup as ML-04/ML-06, kept identical so the before/after comparison in Section 2 is fair.


In [1]:
%pip -q install duckdb huggingface_hub scikit-learn

import os, getpass

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

import duckdb, pandas as pd, numpy as np

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":  f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

MONTH_START = "2026-03-01"
MID_MONTH   = "2026-03-16"
MONTH_END   = "2026-04-01"

print("Connected.")


Connected.


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding A — "The Freshness Multiplier" (Finding 4)

The paper reports that among pages older than a year, the cohort **refreshed in the last 30 days** jumped from 23 to 37 health (**1.6x**) and from 82 to 4.2K impressions (**52x**) versus pages last updated 181–360 days ago.

**My methodology question:** where does the "refreshed" label come from? It looks like refreshing wasn't randomly assigned — an editor or workflow *chose* which 365+ pages to refresh. That means the two groups (refreshed vs stale) could have differed systematically *before* the refresh even happened — for example, if editors tend to prioritize refreshing pages that already show early signs of demand or strategic importance, the 52x gap would partly reflect *which pages get chosen*, not what refreshing itself does. The paper's own Limitations section says "this is a pattern study — we found patterns, not proof of cause and effect," which is the right caveat, but the headline stat cards (1.6x, 52x) are presented without that caveat sitting right next to them. A stronger design would compare pages *matched* on pre-refresh trajectory, or use a true before/after on the same pages rather than two different groups.

This connects directly to my own lane: my ML-05 baseline and ML-06 model both rank *candidates* for review, not proof that acting on them works — the same distinction this finding blurs.

### Finding B — "Which Dead Pages Can Come Back?" (Zombie Recovery model)

The paper reports a model predicting whether a zero-traffic page recovers, scoring **99% same-brand accuracy, 97% on unseen brands** — the highest accuracy of any model in the paper. `Impressions` is listed as the #2 top recovery predictor.

**My methodology question:** a score this high is exactly the shape of result that made me go looking for a second leak in my own ML-04 notebook (where an honest score of 0.999 turned out to be a volume-driven artifact, not real skill). Using `impressions` (whether a page has had any recent impressions) as a *feature* to predict "will this page get impressions again" risks sitting very close to the label's own definition of recovery, rather than genuinely predicting it ahead of time. I'd want to see the exact prediction-time cutoff — were `impressions` measured strictly *before* the recovery window, with a real gap, the same way I had to enforce Mar 1–15 vs Mar 16–31 in my own work? The paper doesn't show that boundary explicitly in the findings page.


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

My ML-06 model was already trained under a client-grouped split — I never actually built the "naive" version to show what it would have looked like without that discipline. Doing that comparison now, as the actual before/after this section asks for: same features, same label, same test size, the only difference is whether the split respects client boundaries.


In [3]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier

honest_features = ["avg_impressions_h1", "avg_clicks_h1", "ctr_h1", "avg_position_h1", "days_with_impressions_h1"]

features = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        AVG(gsc_impressions)                                    AS avg_impressions_h1,
        AVG(gsc_clicks)                                         AS avg_clicks_h1,
        SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0)       AS ctr_h1,
        AVG(gsc_avg_position)                                   AS avg_position_h1,
        COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0) AS days_with_impressions_h1
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{MONTH_START}' AND report_date < DATE '{MID_MONTH}'
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 500
""").df()

labels = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_h2
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{MID_MONTH}' AND report_date < DATE '{MONTH_END}'
    GROUP BY 1, 2
""").df()

data = features.merge(labels, on=["client_hash_id", "content_hash_id"], how="inner")
data["is_declining"] = (data["imp_h2"] < 0.8 * data["avg_impressions_h1"] * 15).astype(int)
data = data.dropna()

def precision_at_k(scores, labels_, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels_)[order[:k]].mean()

K = 50

# BEFORE: naive random split — ignores that pages from the same client can appear in both halves
X, y = data[honest_features], data["is_declining"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
naive_model = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight="balanced", random_state=42, n_jobs=-1)
naive_model.fit(X_tr, y_tr)
naive_p50 = precision_at_k(naive_model.predict_proba(X_te)[:, 1], y_te, K)

# AFTER: client-grouped split — the honest version, same as ML-06
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr_idx, te_idx = next(gss.split(data, groups=data["client_hash_id"]))
train_g, test_g = data.iloc[tr_idx], data.iloc[te_idx]
grouped_model = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight="balanced", random_state=42, n_jobs=-1)
grouped_model.fit(train_g[honest_features], train_g["is_declining"])
grouped_p50 = precision_at_k(grouped_model.predict_proba(test_g[honest_features])[:, 1], test_g["is_declining"], K)

before_after = pd.DataFrame({
    "split": ["BEFORE: naive random split (client leakage possible)", "AFTER: client-grouped split (honest)"],
    "precision_at_50": [round(naive_p50, 3), round(grouped_p50, 3)],
})
before_after


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,split,precision_at_50
0,BEFORE: naive random split (client leakage pos...,0.82
1,AFTER: client-grouped split (honest),0.56


**Reading the before/after:** Yes — the naive random split scored 0.82 vs the grouped split's 0.56, a 0.26-point gap. That gap is the size of the optimism a client-leaky split was buying for free: without grouping, the model could partly "recognize" a client's style from other pages of the same client sitting in both train and test, inflating the apparent score. The honest number for this task is 0.56, not 0.82 — the same shape of problem as my ML-04 leak, just showing up at the split level instead of the feature level.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Re-running the leakage checklist from ML-04 against the exact five features used in my ML-06/ML-09 model.


In [4]:
print("LEAKAGE AUDIT — final feature set")
print("=" * 50)
print(f"Features: {honest_features}")
print(f"Feature window: {MONTH_START} to {MID_MONTH} (first half only)")
print(f"Label window:   {MID_MONTH} to {MONTH_END} (second half only)")
print()

# 1. Are any features calculated after the decision point (Mar 16)?
print("1. All five features are SUM/AVG/COUNT aggregates over report_date < MID_MONTH only.")
print("   Confirmed by construction — the WHERE clause excludes Mar 16 onward.")
print()

# 2. Does the feature window overlap the target window?
overlap_check = con.sql(f"""
    SELECT COUNT(*) AS overlapping_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{MONTH_START}' AND report_date < DATE '{MID_MONTH}'
      AND report_date >= DATE '{MID_MONTH}'
""").df()
print("2. Feature/label window overlap check (should be 0 rows):")
print(overlap_check)
print()

# 3. Re-run the ML-04 leak trap once more on the final feature set, to reconfirm it's clean
leaky_features = honest_features + ["imp_h2"]
data_leak = data.dropna(subset=leaky_features)
Xl, yl = data_leak[leaky_features], data_leak["is_declining"]
Xl_tr, Xl_te, yl_tr, yl_te = train_test_split(Xl, yl, test_size=0.25, random_state=42, stratify=yl)
from sklearn.metrics import roc_auc_score
leaky_check_model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(Xl_tr, yl_tr)
leaky_auc = roc_auc_score(yl_te, leaky_check_model.predict_proba(Xl_te)[:, 1])
print(f"3. Sanity check — re-adding imp_h2 (the label's own source) should spike AUC toward 1.0: {leaky_auc:.3f}")
print(f"   My actual model's honest AUC/Precision stays well below this — confirms the final five features don't smuggle it in.")


LEAKAGE AUDIT — final feature set
Features: ['avg_impressions_h1', 'avg_clicks_h1', 'ctr_h1', 'avg_position_h1', 'days_with_impressions_h1']
Feature window: 2026-03-01 to 2026-03-16 (first half only)
Label window:   2026-03-16 to 2026-04-01 (second half only)

1. All five features are SUM/AVG/COUNT aggregates over report_date < MID_MONTH only.
   Confirmed by construction — the WHERE clause excludes Mar 16 onward.



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2. Feature/label window overlap check (should be 0 rows):
   overlapping_rows
0                 0

3. Sanity check — re-adding imp_h2 (the label's own source) should spike AUC toward 1.0: 0.998
   My actual model's honest AUC/Precision stays well below this — confirms the final five features don't smuggle it in.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original (from ML-06):** *"Model ne baseline ko 2x se zyada beat kar diya (0.28 → 0.66)."*

**Rewritten, safe version:** On this held-out, client-grouped test slice of the March 2026 warehouse data, the Random Forest model's ranking achieved a measured Precision@50 of 0.66, versus 0.28 for the rule-based baseline — a directional, decision-support result observed under one client-grouped split of a single month. It is not yet evidence that this gap holds across other months, the full warehouse, or a live deployment; it tells me this direction is worth carrying into the capstone, not that the number itself is final.


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Two paper findings named with a concrete, constructive methodology question each
- [ ] My own model re-run under a grouped split with a real before/after comparison
- [ ] Leakage audit re-run on the final feature set
- [ ] My boldest claim rewritten in safe, decision-support language
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
